<a href="https://colab.research.google.com/github/nicoavilan/Intro-Comp-Cuant-UNarino-2026/blob/main/Sesion2_Circuitos_Transpilacion_Ruido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introducción a la Computación Cuántica

**Universidad de Nariño — 2026**

Professor: [Nicolás Avilán Vargas](http://www.linkedin.com/in/nicoavilanv)

nicolasg.avilan@urosario.edu.co

**Universidad del Rosario - Escuela de Ciencias e Ingeniería**

**Matemáticas Aplicadas y Ciencias de la Computación**

# Introducción a la Computación Cuántica
## Sesión 2 — Circuitos cuánticos: construcción, compuertas y simulación

### Objetivos de la sesión
Al finalizar esta sesión el participante podrá:
1. Explicar el concepto de universalidad de compuertas y el rol de la transpilación.
2. Usar compuertas paramétricas ($R_x,R_y,R_z$) para preparar estados arbitrarios.
3. Distinguir simulación ideal de simulación con modelo de ruido, e interpretar su efecto sobre los resultados.
4. Construir el circuito de teletransportación cuántica como caso de estudio integrador.

**Prerrequisitos:** contenidos de la Sesión 1 (qubit, superposición, medición, entrelazamiento, compuertas H/X/Z/CNOT).

---
## 0. Preparación del entorno

In [ ]:
%pip install qiskit qiskit-aer qiskit[visualization] --quiet

Esta guía funciona con la versión 2.5.2 de qiskit

In [ ]:
import qiskit
print(qiskit.__version__)

In [ ]:
from qiskit import QuantumCircuit, transpile, ClassicalRegister, QuantumRegister
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.visualization import plot_bloch_multivector, plot_histogram, array_to_latex

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

print("Entorno listo.")


---
## 1. Universalidad de compuertas y transpilación

### 1.1 Conjuntos universales

Un conjunto de compuertas es **universal** si cualquier operador unitario $U$ sobre $n$ qubits puede aproximarse (con precisión arbitraria) mediante una secuencia finita de compuertas de ese conjunto. Ejemplos de conjuntos universales:

$$
\{H, T, \text{CNOT}\}, \qquad \{R_x, R_z, \text{CNOT}\}.
$$

Esto es análogo a la universalidad de $\{\text{NAND}\}$ en computación clásica: un conjunto pequeño y fijo de "piezas" basta para construir cualquier circuito.

### 1.2 ¿Por qué importa esto en la práctica?

El hardware cuántico real solo implementa físicamente un **conjunto nativo** reducido de compuertas (p. ej. $\{R_z, \sqrt{X}, \text{CNOT}\}$ o similar, según el fabricante). Cuando escribimos un circuito con compuertas de "alto nivel" (H, S, T, CCX, ...), el compilador cuántico debe:

1. **Descomponer** cada compuerta en el conjunto nativo del backend.
2. **Enrutar** (*routing*) las interacciones de 2 qubits según la conectividad física del chip (no todos los pares de qubits están directamente conectados).
3. **Optimizar** la secuencia resultante (cancelar compuertas redundantes, fusionar rotaciones).

Este proceso se llama **transpilación**, y en Qiskit se ejecuta con `transpile()`.

In [ ]:
# Circuito de "alto nivel"
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.s(1)
print("Circuito original:")
display(qc.draw(output='mpl'))
print("Numero de compuertas:", qc.size())

In [ ]:
sim = AerSimulator()
qc_transpiled = transpile(qc, sim, optimization_level=1) # Optimización ligera. El más agresivo es 3.
print("Circuito transpilado (optimization_level=1):")
display(qc_transpiled.draw(output='mpl'))
print("Numero de compuertas:", qc_transpiled.size())

Observa que el número de compuertas no cambió — es porque AerSimulator() acepta H, CX y S de forma nativa (a diferencia de un backend real). El efecto de la transpilación se vuelve visible recién cuando restringimos el conjunto de compuertas base, como en la siguiente celda.

In [ ]:
# Efecto de restringir el conjunto de compuertas base
qc_restringido = transpile(qc, basis_gates=['rz', 'sx', 'x', 'cx'], optimization_level=1)
print("Transpilado a base {rz, sx, x, cx}:")
display(qc_restringido.draw(output='mpl'))
print("Numero de compuertas:", qc_restringido.size())

**Observa:** el número de compuertas y su tipo cambian según el conjunto base y el nivel de optimización, pero el operador unitario resultante (salvo fase global) debe seguir representando el mismo circuito lógico.

---
## 2. Compuertas paramétricas: $R_x, R_y, R_z$

Las rotaciones paramétricas generalizan X, Y, Z a ángulos arbitrarios:

$$
R_x(\theta) = e^{-i\theta X/2} = \begin{pmatrix}\cos(\theta/2) & -i\sin(\theta/2) \\ -i\sin(\theta/2) & \cos(\theta/2)\end{pmatrix}, \qquad
R_z(\theta) = e^{-i\theta Z/2} = \begin{pmatrix}e^{-i\theta/2} & 0 \\ 0 & e^{i\theta/2}\end{pmatrix}.
$$

$R_y(\theta)$ es análoga, rotando alrededor del eje $y$. Estas compuertas son la base de los **algoritmos variacionales** (QAOA, VQE, clasificadores cuánticos) que veremos en la Sesión 3: en lugar de compuertas fijas, se usan ángulos $\theta$ como parámetros entrenables.

Cualquier estado de 1 qubit $|\psi(\theta,\phi)\rangle$ de la esfera de Bloch (ver Sesión 1) puede prepararse desde $|0\rangle$ mediante $R_y(\theta)$ seguido de $R_z(\phi)$.

In [ ]:
# Preparamos un estado arbitrario mediante Ry y Rz, y verificamos contra la parametrizacion de Bloch
theta, phi = np.pi/3, np.pi/4

qc_param = QuantumCircuit(1)
qc_param.ry(theta, 0)
qc_param.rz(phi, 0)

sv = Statevector.from_instruction(qc_param)
print("Estado preparado con Ry(theta), Rz(phi):")
display(sv.draw("latex"))

# Comparacion analitica: |psi> = cos(theta/2)|0> + e^{i phi} sin(theta/2)|1>
amp0 = np.cos(theta/2)
amp1 = np.exp(1j*phi) * np.sin(theta/2)
print("Prediccion analitica: [", amp0, ",", amp1, "]")

El resultado no es el mismo al comparar componente a componente los dos resultados. **Verifique** que estos estados difieren en una fase global, por lo que tienen la misma información física.
$$|\psi_{circuito}\rangle = e^{i\gamma}|\psi_{Bloch}\rangle $$

In [ ]:
plot_bloch_multivector(sv)

La representación matricial del circuito se puede obtener como

In [ ]:
Operator(qc_param).draw("latex")

---
## 3. Ruido y decoherencia: por qué importan los simuladores ruidosos

### 3.1 Intuición

Un qubit real no está perfectamente aislado: interactúa con su entorno (campos electromagnéticos, vibraciones térmicas, imperfecciones de fabricación). Esto produce dos efectos principales, caracterizados por tiempos característicos:

- **$T_1$ (relajación de amplitud):** tiempo típico en que un qubit excitado ($|1\rangle$) decae espontáneamente a $|0\rangle$, perdiendo energía hacia el entorno.
- **$T_2$ (decoherencia de fase):** tiempo típico en que se pierde la **fase relativa** entre $\alpha$ y $\beta$, destruyendo la coherencia necesaria para la interferencia cuántica ($T_2 \le 2T_1$).

Cada compuerta y cada operación de medición tiene además una probabilidad de error asociada. El efecto neto es que, en hardware real, el estado deja de ser puro y debe describirse mediante una **matriz de densidad** $\rho$ en vez de un vector $|\psi\rangle$ — no profundizaremos en el formalismo, pero es la razón matemática detrás de por qué "simular con ruido" no es simplemente Statevector + errores aleatorios.

### 3.2 Modelo simplificado: error de despolarización

Un modelo de ruido sencillo y muy usado es el **canal de despolarización**, que con probabilidad $p$ reemplaza el estado por el estado completamente mixto (aleatorio) y con probabilidad $1-p$ lo deja intacto. Es una aproximación conveniente para simular, no una descripción física exacta del hardware.

In [ ]:
# Construccion de un modelo de ruido simple: despolarizacion en compuertas de 1 y 2 qubits
error_1q = depolarizing_error(0.02, 1)   # 2% de error por compuerta de 1 qubit
error_2q = depolarizing_error(0.05, 2)   # 5% de error por compuerta de 2 qubits

noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(error_1q, ['h', 'x', 'ry', 'rz', 'sx']) # (error, instructions)
noise_model.add_all_qubit_quantum_error(error_2q, ['cx'])

print(noise_model)


In [ ]:
# Comparacion: estado de Bell ideal vs. con ruido
qc_bell = QuantumCircuit(2, 2)
qc_bell.h(0)
qc_bell.cx(0, 1)
qc_bell.measure([0,1], [0,1])

sim_ideal = AerSimulator()
sim_noisy = AerSimulator(noise_model=noise_model)

qc_t_ideal = transpile(qc_bell, sim_ideal)
qc_t_noisy = transpile(qc_bell, sim_noisy)

counts_ideal = sim_ideal.run(qc_t_ideal, shots=4000).result().get_counts()
counts_noisy = sim_noisy.run(qc_t_noisy, shots=4000).result().get_counts()

print("Ideal:", counts_ideal)
print("Con ruido:", counts_noisy)

plot_histogram([counts_ideal, counts_noisy], legend=['Ideal', 'Con ruido'],
               title="Estado de Bell: ideal vs. con ruido de despolarizacion")


**Observa:** en el caso ideal solo aparecen `00` y `11`. Con ruido, aparecen también `01` y `10` con probabilidad pequeña pero no nula — estos son "eventos de error" que degradan la fidelidad del estado. Esta es exactamente la clase de efecto que los algoritmos de corrección de errores cuánticos buscan mitigar (tema fuera del alcance de este curso, pero mencionado en la Sesión 3 como parte del panorama NISQ).

---
## 4. Aplicación: teletransportación cuántica

La teletransportación cuántica es un protocolo que transfiere el estado cuántico de un cúbit de una ubicación a otra sin desplazar físicamente el cúbit, utilizando el entrelazamiento cuántico y la comunicación clásica.

Cabe destacar que la teletransportación cuántica se ha demostrado experimentalmente desde 1998. Más recientemente, se han logrado implementaciones exitosas a distancias superiores a los 1.000 kilómetros (resultados experimentales de teletransportación cuántica).



### 4.1 El problema

Alice tiene un qubit en un estado desconocido $|\psi\rangle = \alpha|0\rangle+\beta|1\rangle$ y quiere que Bob lo posea, sin enviarle el qubit físicamente (solo puede enviar 2 bits clásicos) y sin medir $|\psi\rangle$ directamente (lo que destruiría la información, por el teorema de no-clonación). El **protocolo de teletransportación cuántica** logra esto usando un par de Bell compartido de antemano.



![Protocolo de teletransportación cuántica](https://quantum.cloud.ibm.com/learning/images/courses/basics-of-quantum-information/entanglement-in-action/teleportation.svg?dpl=0-1-1564-7f723dc4adad)
*Fuente: IBM Quantum Learning — [Quantum teleportation](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/entanglement-in-action/quantum-teleportation)*

### 4.2 **Protocolo de teletransportación**

La teletransportación cuántica transfiere el estado $|\psi\rangle = \alpha |0\rangle + \beta |1\rangle$ del cúbit de Alice ($q_0$) al cúbit de Bob ($q_2$) utilizando:

*   Un par entrelazado ($|\Phi^+\rangle$) compartido entre Alice ($q_1$) y Bob ($q_2$).
*   Dos bits clásicos para la comunicación.
*   Tres cúbits: $q_0$ (estado a teletransportar), $q_1$ (cúbit entrelazado de Alice), $q_2$ (cúbit entrelazado de Bob).

**Pasos**:
1.  Preparar $|\psi\rangle$ en $q_0$.
2.  Crear $|\Phi^+\rangle$ entre $q_1$ y $q_2$.
3.  Alice realiza una operación CNOT sobre el par ($q_0$, $q_1$), con $q_0$ como control y $q_1$ como objetivo, y luego aplica una operación de Hadamard a $q_0$.
4.  Alice realiza una medición sobre $q_0$ y $q_1$, colapsando su estado a uno de los cuatro estados de Bell y enviando dos bits clásicos a Bob.

Bob aplica puertas condicionales ($ X$, $ Z $) basadas en los bits clásicos para recuperar $|\psi\rangle$  en $ q_2$.

**Descripción matemática**

Estado inicial:
$$|\psi\rangle \otimes |\Phi^+\rangle = (\alpha |0\rangle + \beta |1\rangle)_{q_0} \otimes \frac{1}{\sqrt{2}} (|00\rangle + |11\rangle)_{q_1 q_2}$$

Tras la medición de Alice (CNOT y ( H ), seguidas de la medición), el estado de Bell se proyecta en uno de los siguientes:

$$|00\rangle, |01\rangle, |10\rangle, |11\rangle$$

Bob aplica correcciones de Pauli basadas en los bits clásicos, recuperando $|\psi\rangle$.

**Información cuántica frente a clásica**


*Información cuántica*: El estado $|\psi\rangle$ se transfiere mediante entrelazamiento, no físicamente.

*Información clásica*: Dos bits comunican el resultado de la medición de Alice.

El teorema de no clonación garantiza que $|\psi\rangle$ se destruya en el extremo de Alice.


**Importante:** esto no viola la relatividad ni permite comunicación superlumínica — el paso 4 requiere un canal clásico, limitado por $c$. Tampoco se "clona" el estado: el qubit original de Alice queda destruido (colapsado) tras la medición del paso 3, en consistencia con el teorema de no-clonación.

In [ ]:
def circuito_teletransportacion(theta=None, phi=None):
    """Construye el circuito de teletransportacion.
    Si se dan theta, phi, prepara |psi> = Ry(theta)Rz(phi)|0> en q0 como estado a teletransportar.
    """
    qr = QuantumRegister(3, 'q')   # q0 = psi (Alice), q1 = Alice (mitad Bell), q2 = Bob (mitad Bell)
    cr_alice = ClassicalRegister(2, 'c_alice')
    cr_bob = ClassicalRegister(1, 'c_bob')
    qc = QuantumCircuit(qr, cr_alice, cr_bob)

    # Paso 0: preparar el estado desconocido en q0 (para poder verificar el resultado despues)
    if theta is not None:
        assert phi is not None, "Si se especifica theta, tambien debe especificarse phi"
        qc.ry(theta, 0)
        qc.rz(phi, 0)
    qc.save_statevector(label="inicial")
    qc.barrier()

    # Paso 1: recurso compartido - par de Bell entre q1 y q2
    qc.h(1)
    qc.cx(1, 2)
    qc.barrier()

    # Paso 2: Alice entrelaza q0 con q1
    qc.cx(0, 1)
    qc.h(0)
    qc.barrier()

    # Paso 3: Alice mide q0, q1
    qc.measure(0, cr_alice[0])
    qc.measure(1, cr_alice[1])

    # Paso 5: correcciones condicionadas de Bob sobre q2
    with qc.if_test((cr_alice[1], 1)):
        qc.x(2)
    with qc.if_test((cr_alice[0], 1)):
        qc.z(2)

    return qc


In [ ]:
theta_test, phi_test = np.pi/3, np.pi/5
qc_tp = circuito_teletransportacion(theta_test, phi_test)
print("Estado inicial del sistema completo (3 qubits):")
display(qc_tp.draw(output='mpl'))

In [ ]:
# Verificacion: comparamos el estado final de q2 (Bob) contra el estado original preparado en q0
sim = AerSimulator()

# Ejecutamos y guardamos el statevector final
qc_tp_sv = circuito_teletransportacion(theta_test, phi_test)
qc_tp_sv.save_statevector(label="final")

qc_t_sv = transpile(qc_tp_sv, sim)
result = sim.run(qc_t_sv, shots=1).result()
estado_final = result.data()["final"]

print("Estado final del sistema completo (3 qubits):")
display(estado_final.draw("latex"))

Comparo estados inicial y final

In [ ]:
print("Estado inicial del sistema completo (3 qubits):")
display(result.data()['inicial'].draw("latex"))
print("Estado final del sistema completo (3 qubits):")
display(estado_final.draw("latex"))

In [ ]:
# Estado esperado para q2 aislado (Bob)
qc_ref = QuantumCircuit(1)
qc_ref.ry(theta_test, 0)
qc_ref.rz(phi_test, 0)
psi_esperado = Statevector.from_instruction(qc_ref)
print("\nEstado que Bob deberia tener en q2 (referencia, |psi> original):")
display(psi_esperado.draw("latex"))


In [ ]:
# Verificacion estadistica: repetimos el protocolo y medimos q2 en la base de Bloch
# adecuada para confirmar que reproduce las probabilidades de |psi>

# Version con registro final explicito
qr = QuantumRegister(3, 'q')
c_alice = ClassicalRegister(2, 'c_alice')
c_bob = ClassicalRegister(1, 'c_bob')
qc_full = QuantumCircuit(qr, c_alice, c_bob)
qc_full.ry(theta_test, 0)
qc_full.rz(phi_test, 0)
qc_full.h(1)
qc_full.cx(1, 2)
qc_full.cx(0, 1)
qc_full.h(0)
qc_full.measure(0, c_alice[0])
qc_full.measure(1, c_alice[1])
with qc_full.if_test((c_alice[1], 1)):
    qc_full.x(2)
with qc_full.if_test((c_alice[0], 1)):
    qc_full.z(2)
qc_full.measure(2, c_bob[0])

sim = AerSimulator()
qc_t = transpile(qc_full, sim)
counts = sim.run(qc_t, shots=1000).result().get_counts()

# Extraemos solo el resultado de Bob (c_bob) sumando sobre todos los resultados de Alice
from collections import defaultdict
bob_counts = defaultdict(int)
for outcome, n in counts.items():
    bits = outcome.split()  # 'c_bob c_alice'
    bob_bit = bits[0]
    bob_counts[bob_bit] += n

print("Distribucion de Bob (deberia acercarse a P(0),P(1) de |psi> original):", dict(bob_counts))
p0_esperado = np.cos(theta_test/2)**2
print(f"P(0) esperado analiticamente: {p0_esperado:.3f}")
plot_histogram(dict(bob_counts), title="Resultado en q2 (Bob) tras teletransportacion")


**Conclusión del caso de estudio:** la distribución de probabilidad medida en el qubit de Bob reproduce la del estado original $|\psi\rangle$, confirmando que la información cuántica fue transferida correctamente — usando únicamente entrelazamiento previo, medición local y comunicación clásica de 2 bits. Este circuito integra los cuatro conceptos centrales de la Sesión 1 (superposición, medición, entrelazamiento, compuertas) con las herramientas de simulación de esta sesión.

---
## 5. Resumen de la sesión

| Concepto | Idea clave |
|---|---|
| Universalidad | Un conjunto pequeño de compuertas (p. ej. $\{H,T,\text{CNOT}\}$) genera cualquier unitario |
| Transpilación | Traducción del circuito al conjunto nativo del backend + optimización + *routing* |
| $R_x,R_y,R_z$ | Compuertas paramétricas; base de algoritmos variacionales (Sesión 3) |
| `NoiseModel` | Simulación aproximada del comportamiento de hardware real |
| Teletransportación | Caso de estudio: entrelazamiento + medición + comunicación clásica ⇒ transferencia de un estado cuántico desconocido |

---
## 6. Ejercicios propuestos

### Ejercicio 1 — Transpilación y conteo de compuertas
Construye un circuito de 3 qubits que aplique: $H$ en $q_0$, CNOT($q_0,q_1$), CNOT($q_1,q_2$), $T$ en $q_2$. Transpílalo con `optimization_level=0` y luego con `optimization_level=3`, usando `basis_gates=['rz','sx','x','cx']`. Compara el número de compuertas (`.size()`) y la profundidad (`.depth()`) en cada caso. ¿Qué observas?

In [ ]:
# Ejercicio 1 - tu codigo aqui
qc_ex1 = QuantumCircuit(3)
# completa el circuito descrito




### Ejercicio 2 — Preparación de estados con compuertas paramétricas
Usando solo $R_y$ y $R_z$, escribe una función `preparar_estado(theta, phi)` que devuelva un `QuantumCircuit` de 1 qubit en el estado $|\psi(\theta,\phi)\rangle$ de la Sesión 1. Verifica para al menos 3 pares $(\theta,\phi)$ distintos que el `Statevector` resultante coincide con la fórmula analítica (dentro de una tolerancia numérica pequeña, usa `np.allclose`).

In [ ]:
# Ejercicio 2 - tu codigo aqui

def preparar_estado(theta, phi):
    qc = QuantumCircuit(1)
    # completa
    return qc

# Verifica con np.allclose para 3 pares (theta, phi)



### Ejercicio 3 — Efecto del ruido sobre la fidelidad
Repite la simulación del estado de Bell con `NoiseModel`, pero variando la probabilidad de error de despolarización de las compuertas de 2 qubits: $p \in \{0.0, 0.01, 0.05, 0.1, 0.2\}$. Para cada valor, calcula la **fracción de conteos erróneos** (proporción de resultados `01` + `10` sobre el total). Grafica esta fracción de error en función de $p$ usando `matplotlib`. ¿La relación es lineal?

In [ ]:
# Ejercicio 3 - tu codigo aqui
probabilidades = [0.0, 0.01, 0.05, 0.1, 0.2]
fracciones_error = []

for p in probabilidades:
    # construye el NoiseModel con depolarizing_error(p, 2) para 'cx'
    # ejecuta el circuito de Bell y calcula la fraccion de '01'+'10'
    pass

# plt.plot(probabilidades, fracciones_error, marker='o')
# plt.xlabel("Probabilidad de error de despolarizacion (CNOT)")
# plt.ylabel("Fraccion de resultados erroneos")
# plt.show()


### Ejercicio 4 — Teletransportación con otro estado
Repite la verificación de teletransportación de la Sección 4, pero con el estado $|\psi\rangle = |-\rangle = \frac{|0\rangle-|1\rangle}{\sqrt2}$ (es decir, $\theta=\pi/2,\phi=\pi$). Verifica que Bob recupera exactamente $|-\rangle$, comprobando tanto el `Statevector` final como la distribución de mediciones.

In [ ]:
# Ejercicio 4 - tu codigo aqui
theta_ej4, phi_ej4 = np.pi/2, np.pi
# reutiliza circuito_teletransportacion(theta_ej4, phi_ej4) y verifica




### Ejercicio 5 — Reto (opcional)
¿Qué pasaría si Bob **no** aplica las correcciones condicionadas (pasos con `if_test()`) en el protocolo de teletransportación? Modifica el circuito para omitirlas, ejecuta el protocolo muchas veces, y muestra que el estado de Bob resulta en una **mezcla estadística 50/50** entre $|\psi\rangle$ y su versión corregida — es decir, sin las correcciones clásicas, Bob no puede recuperar el estado original de forma determinista. Relaciona esto con el papel indispensable del canal de comunicación clásico en el protocolo.

In [ ]:
# Ejercicio 5 (reto) - tu codigo aqui




---
## Referencias para profundizar
- Nielsen, M. A. & Chuang, I. L. *Quantum Computation and Quantum Information*, cap. 1.3.7 (teletransportación) y cap. 8 (ruido y canales cuánticos).
- Qiskit documentation — Transpiler: https://docs.quantum.ibm.com/api/qiskit/transpiler
- Qiskit Aer — Noise models: https://qiskit.github.io/qiskit-aer/apidocs/aer_noise.html
- IBM Quantum Learning — módulo sobre teletransportación cuántica: https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/entanglement-in-action/quantum-teleportation


**Para** informar de errores o hacer sugerencias: nicolasg.avilan@urosario.edu.co